In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root       9598 F...m python3
/dev/nvidiactl:      root       9598 F...m python3
/dev/nvidia-uvm:     root       9598 F...m python3


In [2]:
# Check the GPU status
!nvidia-smi

Sun Jul 19 22:19:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             32W /   70W |       0MiB /  15360MiB |     23%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset, Dataset

# Configurations

In [5]:
# Run configuration
LANG = 'en'  # e.g., 'en' | 'ja' | 'id'

# Model configuration
MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Merged-v260711104723'
LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LoRA-v260719113226'
LORA_CKPT_DIR = 'checkpoint-120'
# MODEL_ID = 'FacebookAI/xlm-roberta-base'
# LORA_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-v260711104723'
# LORA_CKPT_DIR = 'checkpoint-320'

# Data configuration
TEST_SIZE = 10
DATA_ID = 'google/xquad'
DATA_DIR = 'xquad.{lang}'
DATA_SPLIT = 'validation'

# Utilities

In [6]:
def load_test_dataset(
    lang, # e.g., 'en' | 'ja' | 'id'
    size,
    data_id=DATA_ID,
    data_dir=DATA_DIR,
    data_split=DATA_SPLIT,
):
    assert '{lang}' in data_dir, "Data directory must contain a '{lang}' placeholder."
    
    dataset_stream = load_dataset(
        data_id,
        data_dir=data_dir.format(lang=lang),
        split=data_split,
        streaming=True,
    )

    test_data = []

    for i, example in enumerate(dataset_stream):
        if i < size:
            test_data.append(example)
        else:
            break

    return Dataset.from_list(test_data)

# Model

In [7]:
# Load the tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')

print()
print(base_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(



XLMRobertaForQuestionAnswering(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768,

In [8]:
# Sanity check
for i in range(11):
    print(f"Base model layer-{i} attention value weight norm:", base_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Base model qa_outputs weight norm:", base_model.qa_outputs.weight.norm().item())
print("Base model qa_outputs bias norm:", base_model.qa_outputs.bias.norm().item())

Base model layer-0 attention value weight norm: 28.367761611938477
Base model layer-1 attention value weight norm: 28.846525192260742
Base model layer-2 attention value weight norm: 30.105730056762695
Base model layer-3 attention value weight norm: 36.34391784667969
Base model layer-4 attention value weight norm: 38.29513931274414
Base model layer-5 attention value weight norm: 41.63006591796875
Base model layer-6 attention value weight norm: 38.20429611206055
Base model layer-7 attention value weight norm: 38.96026611328125
Base model layer-8 attention value weight norm: 39.161903381347656
Base model layer-9 attention value weight norm: 34.90373992919922
Base model layer-10 attention value weight norm: 31.033811569213867

Base model qa_outputs weight norm: 0.9504339098930359
Base model qa_outputs bias norm: 0.010722422040998936


In [ ]:
# Load the LoRA adapter into the base model
lora_model = PeftModel.from_pretrained(base_model, subfolder=LORA_CKPT_DIR, model_id=LORA_ID)
lora_model.eval()

print("device:", lora_model.device)
print()
print(lora_model)

device: cuda:0

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): XLMRobertaForQuestionAnswering(
      (roberta): XLMRobertaModel(
        (embeddings): XLMRobertaEmbeddings(
          (word_embeddings): Embedding(250002, 768, padding_idx=1)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): XLMRobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x XLMRobertaLayer(
              (attention): XLMRobertaAttention(
                (self): XLMRobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
     

In [10]:
# Sanity check
for i in range(11):
    print(f"LoRA model layer-{i} attention value weight norm:", lora_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("LoRA model qa_outputs weight norm:", lora_model.qa_outputs.weight.norm().item())
print("LoRA model qa_outputs bias norm:", lora_model.qa_outputs.bias.norm().item())

LoRA model layer-0 attention value weight norm: 28.367761611938477
LoRA model layer-1 attention value weight norm: 28.846525192260742
LoRA model layer-2 attention value weight norm: 30.105730056762695
LoRA model layer-3 attention value weight norm: 36.34391784667969
LoRA model layer-4 attention value weight norm: 38.29513931274414
LoRA model layer-5 attention value weight norm: 41.63006591796875
LoRA model layer-6 attention value weight norm: 38.20429611206055
LoRA model layer-7 attention value weight norm: 38.96026611328125
LoRA model layer-8 attention value weight norm: 39.161903381347656
LoRA model layer-9 attention value weight norm: 34.90373992919922
LoRA model layer-10 attention value weight norm: 31.033811569213867

LoRA model qa_outputs weight norm: 0.9504339098930359
LoRA model qa_outputs bias norm: 0.010722422040998936


In [11]:
# Sanity check
# merged_qa_outputs_weight = base_model.qa_outputs.weight.detach() + lora_model.qa_outputs.weight.detach()
# merged_qa_outputs_bias = base_model.qa_outputs.bias.detach() + lora_model.qa_outputs.bias.detach()

# print("merged_qa_outputs_weight")
# print("  l2_norm:", merged_qa_outputs_weight.detach().norm().item())
# print("  mean:", merged_qa_outputs_weight.detach().mean().item())
# print("merged_qa_outputs_bias")
# print("  l2_norm:", merged_qa_outputs_bias.detach().norm().item())
# print("  mean:", merged_qa_outputs_bias.detach().mean().item())

In [11]:
# Sanity check
import os
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

def download_hf_model(
        repo_id, 
        ckpt_step, 
        max_checkpoints=10_000,
        ckpt_interval=25,
    ):
    local_dir = repo_id.split('/')[-1]
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) if i != ckpt_step]

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = os.path.join(local_dir, f'checkpoint-{ckpt_step}')
    return local_dir, ckpt_dir

local_dir, ckpt_dir = download_hf_model(
    repo_id=LORA_ID,
    ckpt_step=int(LORA_CKPT_DIR.replace('checkpoint-', '')),
)
lora_state_dict = load_file(os.path.join(ckpt_dir, 'adapter_model.safetensors'))

print(list(lora_state_dict.keys()))

Fetching 197 files:   0%|          | 0/197 [00:00<?, ?it/s]

checkpoint-120/optimizer.pt:   0%|          | 0.00/5.58M [00:00<?, ?B/s]

checkpoint-120/rng_state.pth:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

checkpoint-120/adapter_model.safetensors:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

runs/Jul11_10-48-02_64257332955d/events.(…):   0%|          | 0.00/18.6k [00:00<?, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

training_args.bin:   0%|          | 0.00/5.84k [00:00<?, ?B/s]

['base_model.model.qa_outputs.bias', 'base_model.model.qa_outputs.weight', 'base_model.model.roberta.encoder.layer.0.attention.output.dense.lora_A.weight', 'base_model.model.roberta.encoder.layer.0.attention.output.dense.lora_B.weight', 'base_model.model.roberta.encoder.layer.0.attention.self.key.lora_A.weight', 'base_model.model.roberta.encoder.layer.0.attention.self.key.lora_B.weight', 'base_model.model.roberta.encoder.layer.0.attention.self.query.lora_A.weight', 'base_model.model.roberta.encoder.layer.0.attention.self.query.lora_B.weight', 'base_model.model.roberta.encoder.layer.0.attention.self.value.lora_A.weight', 'base_model.model.roberta.encoder.layer.0.attention.self.value.lora_B.weight', 'base_model.model.roberta.encoder.layer.0.intermediate.dense.lora_A.weight', 'base_model.model.roberta.encoder.layer.0.intermediate.dense.lora_B.weight', 'base_model.model.roberta.encoder.layer.0.output.dense.lora_A.weight', 'base_model.model.roberta.encoder.layer.0.output.dense.lora_B.weight

In [12]:
for n, p in reversed(lora_state_dict.items()):
    # if 'qa_outputs' in n:
    if 'qa_outputs' in n or 'layer.0.intermediate.dense' in n:
        print(n)
        print("  requires_grad:", p.requires_grad)
        print("  l2_norm:", p.detach().norm().item())
        print("  mean:", p.detach().mean().item())

base_model.model.roberta.encoder.layer.0.intermediate.dense.lora_B.weight
  requires_grad: False
  l2_norm: 1.1399354934692383
  mean: 3.2011110306484625e-05
base_model.model.roberta.encoder.layer.0.intermediate.dense.lora_A.weight
  requires_grad: False
  l2_norm: 2.4425551891326904
  mean: -0.000297429010970518
base_model.model.qa_outputs.weight
  requires_grad: False
  l2_norm: 0.9504339098930359
  mean: -0.0005476757069118321
base_model.model.qa_outputs.bias
  requires_grad: False
  l2_norm: 0.010722421109676361
  mean: -0.007573713548481464


# Inference

In [13]:
dataset = load_test_dataset(LANG, size=TEST_SIZE)
print(dataset)

Dataset({
    features: ['id', 'context', 'question', 'answers'],
    num_rows: 10
})


In [14]:
# Inference
for i, example in enumerate(dataset):
    # Tokenize
    inputs = tokenizer(
        example['question'],
        example['context'],
        truncation='only_second',
        max_length=384,
        return_tensors='pt',
    ).to(DEVICE)

    # Predict
    with torch.no_grad():
        outputs = lora_model(**inputs)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

    # Decode answer span
    start_idx = torch.argmax(start_logits)
    end_idx = torch.argmax(end_logits)
    answer_ids = inputs['input_ids'][0, start_idx : end_idx + 1]
    pred_answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    # Ground truth
    gt_answer = example['answers']['text'][0]

    print(f"---- Example {i+1} ----")
    print(f"Question: {example['question']}")
    print(f"Ground truth: {gt_answer}")
    print(f"Predicted: {pred_answer}")
    print()

---- Example 1 ----
Question: How many points did the Panthers defense surrender?
Ground truth: 308
Predicted: 308

---- Example 2 ----
Question: How many career sacks did Jared Allen have?
Ground truth: 136
Predicted: 61⁄2

---- Example 3 ----
Question: How many tackles did Luke Kuechly register?
Ground truth: 118
Predicted: Thomas Davis

---- Example 4 ----
Question: How many balls did Josh Norman intercept?
Ground truth: four
Predicted: 

---- Example 5 ----
Question: Who registered the most sacks on the team this season?
Ground truth: Kawann Short
Predicted: Mario Addison added 61⁄2

---- Example 6 ----
Question: How many interceptions are the Panthers defense credited with in 2015?
Ground truth: 24
Predicted: 24

---- Example 7 ----
Question: Who led the Panthers in sacks?
Ground truth: Kawann Short
Predicted: Kawann Short led the team in sacks with 11

---- Example 8 ----
Question: How many Panthers defense players were selected for the Pro Bowl?
Ground truth: four
Predicted: fou